# Website Traffic & User Journey Analytics

End-to-end analysis of e-commerce website interaction data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', None)
df = pd.read_csv('../data/raw/customer_journey.csv')
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df = df.sort_values(['SessionID','Timestamp']).reset_index(drop=True)
df.head()

In [ ]:
print('Shape:', df.shape)
print('Missing values:\n', df.isnull().sum())
print('Duplicate rows:', df.duplicated().sum())
print('Date range:', df['Timestamp'].min(), 'to', df['Timestamp'].max())

In [ ]:
entry = df.groupby('SessionID')['PageType'].first().rename('EntryPage')
exit_page = df.groupby('SessionID')['PageType'].last().rename('ExitPage')
session_df = (df.groupby('SessionID').agg(UserID=('UserID','first'), StartTime=('Timestamp','min'), EndTime=('Timestamp','max'), DeviceType=('DeviceType','first'), Country=('Country','first'), ReferralSource=('ReferralSource','first'), TotalEvents=('PageType','count'), TotalTimeOnPage=('TimeOnPage_seconds','sum'), MaxItemsInCart=('ItemsInCart','max'), Purchased=('Purchased','max')).reset_index())
session_df['SessionDuration_seconds'] = (session_df['EndTime']-session_df['StartTime']).dt.total_seconds()
session_df['PagesPerSession'] = session_df['TotalEvents']
session_df['Bounced'] = (session_df['PagesPerSession']==1).astype(int)
session_df['Converted'] = (session_df['Purchased']==1).astype(int)
session_df = session_df.merge(entry,on='SessionID').merge(exit_page,on='SessionID')
session_df.head()

In [ ]:
bounce_rate = session_df['Bounced'].mean()*100
conversion_rate = session_df['Converted'].mean()*100
print(f'Bounce rate: {bounce_rate:.2f}%')
print(f'Conversion rate: {conversion_rate:.2f}%')
print(f'Average pages/session: {session_df.PagesPerSession.mean():.2f}')
print(f'Average session duration: {session_df.SessionDuration_seconds.mean():.2f} sec')

In [ ]:
print('Page views:\n', df['PageType'].value_counts())
print('\nEntry pages:\n', session_df['EntryPage'].value_counts())
print('\nExit pages:\n', session_df['ExitPage'].value_counts())

In [ ]:
source_summary = session_df.groupby('ReferralSource').agg(Sessions=('SessionID','count'), BounceRate=('Bounced','mean'), ConversionRate=('Converted','mean')).reset_index()
source_summary['BounceRate']*=100
source_summary['ConversionRate']*=100
source_summary

In [ ]:
daily = df.set_index('Timestamp').resample('D').size()
plt.figure(figsize=(10,5)); plt.plot(daily.index,daily.values); plt.title('Daily Website Interaction Volume'); plt.xlabel('Date'); plt.ylabel('Events'); plt.tight_layout(); plt.show()

In [ ]:
journeys = df.groupby('SessionID')['PageType'].agg(' → '.join).reset_index(name='Journey')
journey_counts = journeys['Journey'].value_counts().reset_index()
journey_counts.columns=['Journey','Sessions']
journey_counts.head(15)

## Business Recommendations

Use the generated summaries and dashboard figures to prioritize high-exit pages, compare acquisition quality, investigate device-specific bounce behavior, and reduce cart/checkout abandonment.